# Urban green accessibility in Turin
## Do residents of Turin have equal access to green spaces?

This notebook analyses the spatial relationship between residential buildings 
and publicly accessible green spaces across Turin's 8 administrative districts (*circoscrizioni*).

**Data sources**
- Overture Maps (2026-02-18) — residential buildings
- OpenStreetMap via Overpass Turbo — parks, gardens, forests, recreation areas
- Comune di Torino — administrative boundaries (EPSG:3003)

**Tools**
- DuckDB + spatial extension — spatial queries and ST_DWithin analysis
- QGIS — green layer extraction and clipping
- Python / Pandas — data wrangling and export

**Methodology notes**
- Green access defined as at least one green space within 300m walking distance
- Analysis does not distinguish between public and private green spaces
- Private gardens and enclosed parks may inflate accessibility figures in hillside districts
- AI assistance: Claude (Anthropic)


In [1]:
BASE_DIR = 'C:/Users/Sonia/Desktop/torino_green/'
SHP_DIR = 'C:/Users/Sonia/Desktop/circoscrizioni_geo/'

In [2]:
import duckdb
import os

In [3]:
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
con.execute ("""
SELECT *
FROM duckdb_extensions ()
WHERE loaded=true;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,extension_name,loaded,installed,install_path,description,aliases,extension_version,install_mode,installed_from
0,core_functions,True,True,(BUILT-IN),Core function library,[],v1.4.4,STATICALLY_LINKED,
1,httpfs,True,True,C:\Users\Sonia\.duckdb\extensions\v1.4.4\windo...,Adds support for reading and writing files ove...,"[http, https, s3]",13f8a81,REPOSITORY,core
2,icu,True,True,(BUILT-IN),Adds support for time zones and collations usi...,[],v1.4.4,STATICALLY_LINKED,
3,json,True,True,(BUILT-IN),Adds support for JSON operations,[],v1.4.4,STATICALLY_LINKED,
4,parquet,True,True,(BUILT-IN),Adds support for reading and writing parquet f...,[],v1.4.4,STATICALLY_LINKED,
5,spatial,True,True,C:\Users\Sonia\.duckdb\extensions\v1.4.4\windo...,Geospatial extension that adds support for wor...,[],f129b24,REPOSITORY,core


In [5]:
con.execute ("SET s3_access_key_id='';")

In [6]:
con.execute("SET s3_secret_access_key ='';")

In [7]:
df_building_torino = con.execute ("""
SELECT id, subtype, class,
ST_AsText (geometry) AS geom_wkt
FROM read_parquet ('s3://overturemaps-us-west-2/release/2026-02-18.0/theme=buildings/type=building/*',hive_partitioning=false)
WHERE subtype='residential'
AND class NOT IN ('garage', 'garages', 'parking')
AND bbox.xmin BETWEEN 7.6 AND 7.72
AND bbox.ymin BETWEEN 44.99 AND 45.12
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
os.makedirs(f'{BASE_DIR}', exist_ok=True)
print(os.path.exists(f'{BASE_DIR}'))

True


In [ ]:
df_building_torino.to_parquet(f'{BASE_DIR}buildings_raw.parquet')
print("salvato")

In [9]:
print(df_building_torino.shape)
print(df_building_torino.columns.tolist())
print(df_building_torino.head())

(18692, 4)
['id', 'subtype', 'class', 'geom_wkt']
                                     id      subtype       class  \
0  3d757765-5fe7-4b2b-8687-00c42fefee09  residential  apartments   
1  3607c00c-7917-44f8-8614-ca836de2c3fd  residential  apartments   
2  b739c849-cf98-48e1-997b-13ba1fbdb4dd  residential  apartments   
3  be5309d2-60ff-4099-be0a-4af86194cd25  residential  apartments   
4  be4d5140-98e4-48a2-adaa-35c0a80b1e1e  residential  apartments   

                                            geom_wkt  
0  POLYGON ((7.7132242 44.9931495, 7.71275 44.993...  
1  POLYGON ((7.7136297 44.9934335, 7.7134365 44.9...  
2  POLYGON ((7.7133559 44.9936608, 7.7132624 44.9...  
3  POLYGON ((7.7137459 44.9937864, 7.7136272 44.9...  
4  POLYGON ((7.713125 44.9935223, 7.7131899 44.99...  


In [10]:
torino_boundary = con.execute (f"""
SELECT ST_AsText (
ST_Transform (
ST_Union_Agg(geom),
'EPSG:3003',
'EPSG:4326',
true
)
) AS geom_wkt
FROM ST_ReadSHP ('{SHP_DIR}circoscrizioni_geo.shp', encoding='UTF-8')
""").df()


In [11]:
print(torino_boundary['geom_wkt'].iloc[0][:100])

MULTIPOLYGON (((7.581267675773978 45.04397119554554, 7.581412737404142 45.04395561331498, 7.58168418


In [12]:
con.register ('buildings', df_building_torino)

In [13]:
con.register ('boundary', torino_boundary)

In [14]:
df_buildings_filtered = con.execute("""
    SELECT b.id, b.subtype, b.class, b.geom_wkt
    FROM buildings b, boundary bo
    WHERE ST_Within(
        ST_GeomFromText(b.geom_wkt),
        ST_GeomFromText(bo.geom_wkt)
    )
""").df()

print(len(df_buildings_filtered))

17178


In [15]:
df_buildings_filtered.to_parquet(f'{BASE_DIR}buildings_filtered.parquet')
torino_boundary.to_csv(f'{BASE_DIR}boundary.csv', index=False)
print("salvati")

salvati


In [16]:
# per sessioni successive senza rieseguire S3, decommenta:
# import pandas as pd
# df_buildings_filtered = pd.read_parquet(f'{BASE_DIR}buildings_filtered.parquet')
# torino_boundary = pd.read_csv(f'{BASE_DIR}boundary.csv')
print(len(df_buildings_filtered))
print(torino_boundary['geom_wkt'].iloc[0][:100])

17178
MULTIPOLYGON (((7.581267675773978 45.04397119554554, 7.581412737404142 45.04395561331498, 7.58168418


In [17]:
df_verde = con.execute(f"""
    SELECT ST_AsText(geom) AS geom_wkt
    FROM ST_Read('{BASE_DIR}verde-torino.geojson')
""").df()

In [18]:
con.register('buildings', df_buildings_filtered)
con.register('verde', df_verde)

df_access = con.execute("""
    SELECT 
        b.id,
        b.subtype,
        b.class,
        b.geom_wkt,
        CASE WHEN COUNT(v.geom_wkt) > 0 THEN true ELSE false END AS has_green_access
    FROM buildings b
    LEFT JOIN verde v
        ON ST_DWithin(
            ST_Transform(ST_GeomFromText(b.geom_wkt), 'EPSG:4326', 'EPSG:32632', true),
            ST_Transform(ST_GeomFromText(v.geom_wkt), 'EPSG:4326', 'EPSG:32632', true),
            300
        )
    GROUP BY b.id, b.subtype, b.class, b.geom_wkt
""").df()

print(len(df_access))
print(df_access['has_green_access'].value_counts())

17178
has_green_access
True     11167
False     6011
Name: count, dtype: int64


In [19]:
df_circoscrizioni = con.execute(f"""
    SELECT 
        NCIRCO,
        DENOM,
        ST_AsText(
            ST_Transform(geom, 'EPSG:3003', 'EPSG:4326', true)
        ) AS geom_wkt
    FROM ST_ReadSHP('{SHP_DIR}circoscrizioni_geo.shp', encoding='UTF-8')
""").df()

print(len(df_circoscrizioni))
print(df_circoscrizioni[['NCIRCO', 'DENOM']])

8
   NCIRCO                                              DENOM
0       1                                  Centro - Crocetta
1       2                             Santa Rita - Mirafiori
2       3  San Paolo - Cenisia - Pozzo Strada - Cit Turin...
3       5  Borgo Vittoria - Madonna di Campagna - Lucento...
4       6  Barriera di Milano - Regio Parco - Barca - Ber...
5       7    Aurora, Vanchiglia - Sassi - Madonna del Pilone
6       8  San Salvario - Cavoretto - Borgo Po - Nizza Mi...
7       4                 San Donato - Campidoglio - Parella


In [20]:
con.register('circoscrizioni', df_circoscrizioni)
con.register('access', df_access)

df_result = con.execute("""
    SELECT 
        c.NCIRCO,
        c.DENOM,
        COUNT(a.id) AS total_buildings,
        SUM(CASE WHEN a.has_green_access THEN 1 ELSE 0 END) AS buildings_with_green,
        ROUND(100.0 * SUM(CASE WHEN a.has_green_access THEN 1 ELSE 0 END) / COUNT(a.id), 1) AS pct_green_access
    FROM access a
    JOIN circoscrizioni c
        ON ST_Within(
            ST_GeomFromText(a.geom_wkt),
            ST_GeomFromText(c.geom_wkt)
        )
    GROUP BY c.NCIRCO, c.DENOM
    ORDER BY pct_green_access ASC
""").df()

print(df_result)

   NCIRCO                                              DENOM  total_buildings  \
0       5  Borgo Vittoria - Madonna di Campagna - Lucento...              833   
1       6  Barriera di Milano - Regio Parco - Barca - Ber...              844   
2       7    Aurora, Vanchiglia - Sassi - Madonna del Pilone              875   
3       8  San Salvario - Cavoretto - Borgo Po - Nizza Mi...             2961   
4       4                 San Donato - Campidoglio - Parella             3005   
5       1                                  Centro - Crocetta             2285   
6       3  San Paolo - Cenisia - Pozzo Strada - Cit Turin...             4191   
7       2                             Santa Rita - Mirafiori             2184   

   buildings_with_green  pct_green_access  
0                  41.0               4.9  
1                  69.0               8.2  
2                 489.0              55.9  
3                1691.0              57.1  
4                1779.0              59.2  
5     

In [21]:
df_access.to_parquet(f'{BASE_DIR}access.parquet')
df_result.to_parquet(f'{BASE_DIR}result_by_circoscrizione.parquet')
print("salvati")

salvati


In [22]:
con.register('result_by_circoscrizione', df_result)

df_result_geo = con.execute("""
    SELECT 
        r.NCIRCO,
        r.DENOM,
        r.total_buildings,
        r.buildings_with_green,
        r.pct_green_access,
        c.geom_wkt
    FROM result_by_circoscrizione r
    JOIN circoscrizioni c ON r.NCIRCO = c.NCIRCO
""").df()

print(df_result_geo.shape)

(8, 6)


In [24]:
import geopandas as gpd
from shapely import wkt

gdf = gpd.GeoDataFrame(
    df_result_geo,
    geometry=df_result_geo['geom_wkt'].apply(wkt.loads),
    crs='EPSG:4326'
)
gdf.to_file(f'{BASE_DIR}result_circoscrizioni.geojson', driver='GeoJSON')
print("salvato")

salvato
